# Notebook 06 — Fraud Risk Score Explainability

## Fraud Graph Analytics  
### Score Final de Risco Antifraude com Explicabilidade Transacional e Relacional

Este notebook combina o motor de regras antifraude desenvolvido no **Notebook 03** com as features estruturais de grafo geradas no **Notebook 05**.

O objetivo é criar um score final de risco antifraude, combinando:

- score de regras transacionais;
- risco estrutural da conta;
- risco estrutural do dispositivo;
- risco estrutural do beneficiário;
- risco estrutural do IP;
- risco da comunidade de grafo;
- explicações textuais para apoiar investigação.

A proposta é demonstrar como regras explicáveis e Graph Analytics podem ser integrados em uma camada de priorização operacional para prevenção a fraudes.

## 1. Objetivo da Célula

### Objetivo

Configurar o ambiente inicial, carregar os artefatos dos notebooks anteriores e preparar os dados para composição do score final.

### Ações realizadas

- Importação das bibliotecas principais.
- Definição dos diretórios do projeto.
- Carregamento da base transacional com score de regras.
- Carregamento das features estruturais de grafo.
- Carregamento do ranking de entidades.
- Preparação dos diretórios de saída.

### Justificativa técnica

O score final deve combinar sinais transacionais e relacionais. O motor de regras captura eventos suspeitos na transação, enquanto o grafo captura contexto estrutural, como conectividade, comunidades, centralidade e compartilhamento de entidades.

### Resultados esperados

Bases carregadas e preparadas para enriquecimento, cálculo do score final e geração de explicações antifraude.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
pd.set_option("display.max_columns", 180)
pd.set_option("display.max_colwidth", 220)
pd.set_option("display.float_format", "{:,.4f}".format)

In [3]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
GOLD_DIR = DATA_DIR / "03-gold"

DOCS_DIR = PROJECT_ROOT / "docs"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = ARTIFACTS_DIR / "reports"

GOLD_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Gold dir:     {GOLD_DIR}")
print(f"Docs dir:     {DOCS_DIR}")
print(f"Reports dir:  {REPORTS_DIR}")

Project root: d:\_DS-Projects\Data-Science\fraud-graph-analytics
Gold dir:     d:\_DS-Projects\Data-Science\fraud-graph-analytics\data\03-gold
Docs dir:     d:\_DS-Projects\Data-Science\fraud-graph-analytics\docs
Reports dir:  d:\_DS-Projects\Data-Science\fraud-graph-analytics\artifacts\reports


## 2. Carregamento dos Artefatos Analíticos

Nesta etapa serão carregadas as principais saídas dos notebooks anteriores:

- `transactions_with_rule_scores.parquet`;
- `antifraud_alerts.parquet`;
- `graph_entity_features.parquet`;
- `graph_entity_ranking.parquet`;
- `community_risk_summary.parquet`.

Esses artefatos representam a união entre a camada transacional, o motor de regras e a camada relacional de grafo.

In [4]:
required_files = {
    "rules_scored": GOLD_DIR / "transactions_with_rule_scores.parquet",
    "alerts": GOLD_DIR / "antifraud_alerts.parquet",
    "graph_entity_features": GOLD_DIR / "graph_entity_features.parquet",
    "graph_entity_ranking": GOLD_DIR / "graph_entity_ranking.parquet",
    "community_risk_summary": GOLD_DIR / "community_risk_summary.parquet",
}

for name, path in required_files.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Arquivo obrigatório não encontrado: {path}. "
            "Verifique se os notebooks anteriores foram executados."
        )

rules_scored = pd.read_parquet(required_files["rules_scored"])
alerts = pd.read_parquet(required_files["alerts"])
graph_entity_features = pd.read_parquet(required_files["graph_entity_features"])
graph_entity_ranking = pd.read_parquet(required_files["graph_entity_ranking"])
community_risk_summary = pd.read_parquet(required_files["community_risk_summary"])

rules_scored["data_hora"] = pd.to_datetime(rules_scored["data_hora"])
alerts["data_hora"] = pd.to_datetime(alerts["data_hora"])

print("Artefatos carregados com sucesso.")

Artefatos carregados com sucesso.


In [5]:
loaded_summary = pd.DataFrame(
    [
        {
            "artefato": name,
            "linhas": df.shape[0],
            "colunas": df.shape[1],
            "memoria_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
        }
        for name, df in {
            "rules_scored": rules_scored,
            "alerts": alerts,
            "graph_entity_features": graph_entity_features,
            "graph_entity_ranking": graph_entity_ranking,
            "community_risk_summary": community_risk_summary,
        }.items()
    ]
)

loaded_summary

,artefato,linhas,colunas,memoria_mb
0,rules_scored,80000,82,174.5600
1,alerts,79997,33,83.0800
2,graph_entity_features,17000,32,8.4400
3,graph_entity_ranking,17000,19,5.7800
4,community_risk_summary,21,18,0.0000


## 3. Preparação das Features de Grafo por Entidade

As features de grafo foram geradas por entidade no Notebook 05.

Agora serão criados subconjuntos específicos para:

- conta;
- dispositivo;
- beneficiário;
- IP.

Cada subconjunto será renomeado com prefixos específicos para evitar ambiguidade durante os joins.

In [6]:
graph_feature_cols = [
    "entity_id",
    "degree",
    "weighted_degree",
    "pagerank",
    "betweenness_approx",
    "component_id",
    "component_size",
    "community_id",
    "community_size",
    "community_risk_score",
    "qtd_alertas",
    "score_medio",
    "score_max",
    "taxa_alerta",
    "taxa_fraude_sintetica",
    "valor_total",
    "entity_graph_risk_score",
    "entity_graph_risk_band",
]


def build_entity_features(df: pd.DataFrame, node_type: str, prefix: str) -> pd.DataFrame:
    temp = (
        df.loc[df["node_type"] == node_type, graph_feature_cols]
        .drop_duplicates("entity_id")
        .copy()
    )

    rename_map = {
        col: f"{prefix}_{col}"
        for col in temp.columns
        if col != "entity_id"
    }

    temp = temp.rename(columns=rename_map)

    return temp


account_graph_features = build_entity_features(graph_entity_features, "Conta", "account_graph")
device_graph_features = build_entity_features(graph_entity_features, "Dispositivo", "device_graph")
beneficiary_graph_features = build_entity_features(graph_entity_features, "Beneficiario", "beneficiary_graph")
ip_graph_features = build_entity_features(graph_entity_features, "IP", "ip_graph")

display(account_graph_features.head())
display(device_graph_features.head())
display(beneficiary_graph_features.head())
display(ip_graph_features.head())

,entity_id,account_graph_degree,account_graph_weighted_degree,account_graph_pagerank,account_graph_betweenness_approx,account_graph_component_id,account_graph_component_size,account_graph_community_id,account_graph_community_size,account_graph_community_risk_score,account_graph_qtd_alertas,account_graph_score_medio,account_graph_score_max,account_graph_taxa_alerta,account_graph_taxa_fraude_sintetica,account_graph_valor_total,account_graph_entity_graph_risk_score,account_graph_entity_graph_risk_band
0,CTA_000001,42,343.8000,0.0001,0.0002,1,17000,2,1571,77.0100,14,41.8571,75,1.0000,0.0714,"11,352.5400",73.6100,alto
15,CTA_000002,36,306.0000,0.0001,0.0001,1,17000,14,915,70.4300,12,45.0000,60,1.0000,0.1667,"8,233.2000",59.9900,medio
28,CTA_000003,42,369.6000,0.0001,0.0001,1,17000,13,768,62.1800,14,48.0000,82,1.0000,0.0000,"4,974.4500",68.4000,alto
43,CTA_000004,39,337.2000,0.0001,0.0001,1,17000,12,1320,74.7500,13,46.4615,66,1.0000,0.0769,"4,242.8000",64.7800,alto
57,CTA_000005,36,324.0000,0.0001,0.0000,1,17000,6,650,53.0100,12,50.0000,66,1.0000,0.1667,"6,293.0000",55.3100,medio


,entity_id,device_graph_degree,device_graph_weighted_degree,device_graph_pagerank,device_graph_betweenness_approx,device_graph_component_id,device_graph_component_size,device_graph_community_id,device_graph_community_size,device_graph_community_risk_score,device_graph_qtd_alertas,device_graph_score_medio,device_graph_score_max,device_graph_taxa_alerta,device_graph_taxa_fraude_sintetica,device_graph_valor_total,device_graph_entity_graph_risk_score,device_graph_entity_graph_risk_band
1,DEV_000478,18,148.6000,0.0000,0.0000,1,17000,16,1914,82.0100,18,42.5556,66,1.0000,0.0000,"7,540.8300",34.3400,baixo
2,DEV_000789,15,120.8000,0.0000,0.0000,1,17000,2,1571,77.0100,15,40.5333,52,1.0000,0.0000,"4,300.9000",21.3600,baixo
3,DEV_001310,24,200.7000,0.0000,0.0001,1,17000,16,1914,82.0100,24,43.6250,87,1.0000,0.0833,"25,462.1100",57.5500,medio
4,DEV_001330,15,130.7000,0.0000,0.0000,1,17000,14,915,70.4300,15,47.1333,97,1.0000,0.0667,"7,695.7900",27.2100,baixo
5,DEV_001375,21,180.0000,0.0000,0.0000,1,17000,19,401,36.3100,21,45.7143,66,1.0000,0.0000,"6,149.4100",37.4100,medio


,entity_id,beneficiary_graph_degree,beneficiary_graph_weighted_degree,beneficiary_graph_pagerank,beneficiary_graph_betweenness_approx,beneficiary_graph_component_id,beneficiary_graph_component_size,beneficiary_graph_community_id,beneficiary_graph_community_size,beneficiary_graph_community_risk_score,beneficiary_graph_qtd_alertas,beneficiary_graph_score_medio,beneficiary_graph_score_max,beneficiary_graph_taxa_alerta,beneficiary_graph_taxa_fraude_sintetica,beneficiary_graph_valor_total,beneficiary_graph_entity_graph_risk_score,beneficiary_graph_entity_graph_risk_band
10500,BEN_000122,26,258.9000,0.0001,0.0000,1,17000,12,1320,74.7500,26,59.5769,100,1.0000,0.0769,"13,296.4000",64.2300,alto
10501,BEN_000207,24,183.2000,0.0000,0.0001,1,17000,5,562,46.6600,24,36.3333,100,1.0000,0.0417,"10,063.3400",52.7300,medio
10502,BEN_000323,25,242.2000,0.0001,0.0001,1,17000,4,1326,73.5800,25,56.8800,100,1.0000,0.0800,"53,146.9100",61.8100,alto
10503,BEN_000332,26,244.0000,0.0001,0.0000,1,17000,2,1571,77.0100,26,53.8462,66,1.0000,0.0000,"10,723.9900",60.2000,alto
10504,BEN_000452,30,284.7000,0.0001,0.0000,1,17000,15,370,30.3000,30,54.9000,97,1.0000,0.0333,"12,966.5300",62.6700,alto


,entity_id,ip_graph_degree,ip_graph_weighted_degree,ip_graph_pagerank,ip_graph_betweenness_approx,ip_graph_component_id,ip_graph_component_size,ip_graph_community_id,ip_graph_community_size,ip_graph_community_risk_score,ip_graph_qtd_alertas,ip_graph_score_medio,ip_graph_score_max,ip_graph_taxa_alerta,ip_graph_taxa_fraude_sintetica,ip_graph_valor_total,ip_graph_entity_graph_risk_score,ip_graph_entity_graph_risk_band
14000,IP_000123,41,330.2000,0.0001,0.0002,1,17000,9,496,45.8200,41,40.5366,86,1.0000,0.0488,"16,820.1500",77.1100,alto
14001,IP_000186,30,240.6000,0.0001,0.0004,1,17000,2,1571,77.0100,30,40.2000,82,1.0000,0.0667,"11,832.3200",72.6800,alto
14002,IP_000447,38,326.3000,0.0001,0.0003,1,17000,14,915,70.4300,38,45.8684,97,1.0000,0.0263,"15,273.8900",82.3500,critico
14003,IP_000541,22,175.2000,0.0000,0.0001,1,17000,2,1571,77.0100,22,39.6364,60,1.0000,0.0000,"8,974.1600",45.9800,medio
14004,IP_000919,25,205.2000,0.0001,0.0002,1,17000,17,722,53.0700,26,40.5385,66,1.0000,0.0769,"11,992.1000",54.9800,medio


## 4. Enriquecimento Transacional com Features de Grafo

Nesta etapa, cada transação será enriquecida com informações estruturais das entidades conectadas a ela:

- risco de grafo da conta de origem;
- risco de grafo do dispositivo;
- risco de grafo do beneficiário;
- risco de grafo do IP;
- risco das comunidades relacionadas.

Esse enriquecimento permite que o score final combine sinais da transação e sinais da rede.

In [7]:
score_base = (
    rules_scored
    .merge(
        account_graph_features,
        left_on="conta_origem_id",
        right_on="entity_id",
        how="left",
    )
    .drop(columns=["entity_id"])
    .merge(
        device_graph_features,
        left_on="device_id",
        right_on="entity_id",
        how="left",
    )
    .drop(columns=["entity_id"])
    .merge(
        beneficiary_graph_features,
        left_on="beneficiario_id",
        right_on="entity_id",
        how="left",
    )
    .drop(columns=["entity_id"])
    .merge(
        ip_graph_features,
        left_on="ip_id",
        right_on="entity_id",
        how="left",
    )
    .drop(columns=["entity_id"])
)

score_base.head()

,transacao_id,conta_origem_id,beneficiario_id,valor,data_hora,tipo_transacao,canal,device_id,ip_id,status_transacao,is_fraud,fraud_scenario,cartao_id,conta_id,cliente_id,tipo_conta,data_abertura,status_conta,limite_transacional_diario,idade,uf,segmento,data_cadastro,score_cadastral,tipo_device,sistema_operacional,fingerprint_risco,uf_origem,tipo_rede,risco_rede,tipo_beneficiario,banco_destino,uf_destino,idade_conta_dias,hora,data,ano_mes,data_hora_hora,valor_sobre_limite_diario,device_qtd_transacoes,device_contas_distintas,device_clientes_distintos,device_beneficiarios_distintos,device_valor_total,device_valor_medio,device_taxa_fraude_sintetica,benef_qtd_transacoes,benef_contas_origem_distintas,benef_clientes_distintos,benef_devices_distintos,benef_valor_total,benef_valor_medio,benef_taxa_fraude_sintetica,ip_qtd_transacoes,ip_contas_distintas,ip_clientes_distintos,ip_devices_distintos,ip_valor_total,ip_taxa_fraude_sintetica,qtd_transacoes_conta_hora,valor_total_conta_hora,qtd_beneficiarios_conta_hora,qtd_transacoes_conta_dia,valor_total_conta_dia,qtd_beneficiarios_conta_dia,qtd_devices_conta_dia,R001_alto_valor_transacional,R002_valor_acima_limite_diario,R003_conta_nova_alto_valor,R004_dispositivo_compartilhado,R005_beneficiario_concentrador,R006_rede_ou_device_alto_risco,R007_rajada_transacional_horaria,R008_muitos_beneficiarios_no_dia,R009_ip_compartilhado_multiplas_contas,R010_canal_digital_alto_valor,qtd_regras_acionadas,rule_score_raw,rule_score,risk_band,alerta_gerado,alert_explanation,account_graph_degree,account_graph_weighted_degree,account_graph_pagerank,account_graph_betweenness_approx,account_graph_component_id,account_graph_component_size,account_graph_community_id,account_graph_community_size,account_graph_community_risk_score,account_graph_qtd_alertas,account_graph_score_medio,account_graph_score_max,account_graph_taxa_alerta,account_graph_taxa_fraude_sintetica,account_graph_valor_total,account_graph_entity_graph_risk_score,account_graph_entity_graph_risk_band,device_graph_degree,device_graph_weighted_degree,device_graph_pagerank,device_graph_betweenness_approx,device_graph_component_id,device_graph_component_size,device_graph_community_id,device_graph_community_size,device_graph_community_risk_score,device_graph_qtd_alertas,device_graph_score_medio,device_graph_score_max,device_graph_taxa_alerta,device_graph_taxa_fraude_sintetica,device_graph_valor_total,device_graph_entity_graph_risk_score,device_graph_entity_graph_risk_band,beneficiary_graph_degree,beneficiary_graph_weighted_degree,beneficiary_graph_pagerank,beneficiary_graph_betweenness_approx,beneficiary_graph_component_id,beneficiary_graph_component_size,beneficiary_graph_community_id,beneficiary_graph_community_size,beneficiary_graph_community_risk_score,beneficiary_graph_qtd_alertas,beneficiary_graph_score_medio,beneficiary_graph_score_max,beneficiary_graph_taxa_alerta,beneficiary_graph_taxa_fraude_sintetica,beneficiary_graph_valor_total,beneficiary_graph_entity_graph_risk_score,beneficiary_graph_entity_graph_risk_band,ip_graph_degree,ip_graph_weighted_degree,ip_graph_pagerank,ip_graph_betweenness_approx,ip_graph_component_id,ip_graph_component_size,ip_graph_community_id,ip_graph_community_size,ip_graph_community_risk_score,ip_graph_qtd_alertas,ip_graph_score_medio,ip_graph_score_max,ip_graph_taxa_alerta,ip_graph_taxa_fraude_sintetica,ip_graph_valor_total,ip_graph_entity_graph_risk_score,ip_graph_entity_graph_risk_band
0,TX_00007707,CTA_001877,BEN_003322,203.3000,2025-01-01 03:05:51,pix,app,DEV_001817,IP_000089,aprovada,0,normal,None,CTA_001877,CLI_002620,corrente,2021-08-09,ativa,5000,74,MG,varejo,2025-09-11,803,mobile,iOS,baixo,CE,movel,alto,pessoa_fisica,banco_a,GO,1241,3,2025-01-01,2025-01,2025-01-01 03:00:00,0.0407,21,21,21,21,"8,564.2800",407.8229,0.0000,24,24,24,24,"8,964.8900",373.5371,0.0417,31,31,31,31,"13,928.8700",0.0968,1,203.3000,1,1,203.3000,1,1,False,False,False,True,False,True,False,False,True,False,3,46,46,alto,True,Score 46/100 — risco

In [8]:
graph_score_cols = [
    "account_graph_entity_graph_risk_score",
    "device_graph_entity_graph_risk_score",
    "beneficiary_graph_entity_graph_risk_score",
    "ip_graph_entity_graph_risk_score",
    "account_graph_community_risk_score",
    "device_graph_community_risk_score",
    "beneficiary_graph_community_risk_score",
    "ip_graph_community_risk_score",
]

for col in graph_score_cols:
    if col not in score_base.columns:
        score_base[col] = 0

score_base[graph_score_cols] = score_base[graph_score_cols].fillna(0)

score_base["max_entity_graph_risk_score"] = score_base[
    [
        "account_graph_entity_graph_risk_score",
        "device_graph_entity_graph_risk_score",
        "beneficiary_graph_entity_graph_risk_score",
        "ip_graph_entity_graph_risk_score",
    ]
].max(axis=1)

score_base["max_community_risk_score"] = score_base[
    [
        "account_graph_community_risk_score",
        "device_graph_community_risk_score",
        "beneficiary_graph_community_risk_score",
        "ip_graph_community_risk_score",
    ]
].max(axis=1)

score_base[
    [
        "transacao_id",
        "rule_score",
        "account_graph_entity_graph_risk_score",
        "device_graph_entity_graph_risk_score",
        "beneficiary_graph_entity_graph_risk_score",
        "ip_graph_entity_graph_risk_score",
        "max_entity_graph_risk_score",
        "max_community_risk_score",
    ]
].head()

,transacao_id,rule_score,account_graph_entity_graph_risk_score,device_graph_entity_graph_risk_score,beneficiary_graph_entity_graph_risk_score,ip_graph_entity_graph_risk_score,max_entity_graph_risk_score,max_community_risk_score
0,TX_00007707,46,68.8800,34.6600,45.1200,63.0400,68.8800,82.0100
1,TX_00072064,32,70.6500,73.6600,48.4600,60.5000,73.6600,82.0100
2,TX_00075764,52,70.5900,30.6400,55.6200,48.8100,70.5900,77.0100
3,TX_00023203,32,73.6700,46.1500,52.3700,47.7600,73.6700,66.5100
4,TX_00001302,52,77.1000,27.7500,74.1100,56.2900,77.1000,82.0100


## 5. Metodologia do Score Final

O score final será calculado combinando sinais transacionais e relacionais.

### Componentes

| Componente | Peso | Interpretação |
|---|---:|---|
| `rule_score` | 45% | Risco transacional baseado em regras explicáveis |
| `account_graph_risk` | 20% | Risco estrutural da conta |
| `device_graph_risk` | 12,5% | Risco estrutural do dispositivo |
| `beneficiary_graph_risk` | 12,5% | Risco estrutural do beneficiário |
| `ip_graph_risk` | 5% | Risco estrutural do IP |
| `max_community_risk` | 5% | Risco máximo das comunidades associadas |

A lógica é priorizar o comportamento transacional, mas enriquecer o score com contexto relacional.

In [9]:
score_weights = {
    "rule_score": 0.45,
    "account_graph_entity_graph_risk_score": 0.20,
    "device_graph_entity_graph_risk_score": 0.125,
    "beneficiary_graph_entity_graph_risk_score": 0.125,
    "ip_graph_entity_graph_risk_score": 0.05,
    "max_community_risk_score": 0.05,
}

score_methodology = pd.DataFrame(
    [
        {
            "componente": component,
            "peso": weight,
            "peso_percentual": f"{weight:.1%}",
        }
        for component, weight in score_weights.items()
    ]
)

score_methodology

,componente,peso,peso_percentual
0,rule_score,0.4500,45.0%
1,account_graph_entity_graph_risk_score,0.2000,20.0%
2,device_graph_entity_graph_risk_score,0.1250,12.5%
3,beneficiary_graph_entity_graph_risk_score,0.1250,12.5%
4,ip_graph_entity_graph_risk_score,0.0500,5.0%
5,max_community_risk_score,0.0500,5.0%


## 6. Cálculo do Fraud Risk Score

Nesta etapa será calculado o score final.

O score final será limitado ao intervalo de 0 a 100 e classificado em quatro faixas:

| Faixa | Critério |
|---|---|
| `critico` | score >= 80 |
| `alto` | 60 <= score < 80 |
| `medio` | 35 <= score < 60 |
| `baixo` | score < 35 |

In [10]:
score_base["fraud_risk_score_raw"] = (
    score_weights["rule_score"] * score_base["rule_score"].fillna(0)
    + score_weights["account_graph_entity_graph_risk_score"]
    * score_base["account_graph_entity_graph_risk_score"].fillna(0)
    + score_weights["device_graph_entity_graph_risk_score"]
    * score_base["device_graph_entity_graph_risk_score"].fillna(0)
    + score_weights["beneficiary_graph_entity_graph_risk_score"]
    * score_base["beneficiary_graph_entity_graph_risk_score"].fillna(0)
    + score_weights["ip_graph_entity_graph_risk_score"]
    * score_base["ip_graph_entity_graph_risk_score"].fillna(0)
    + score_weights["max_community_risk_score"]
    * score_base["max_community_risk_score"].fillna(0)
)

score_base["fraud_risk_score"] = score_base["fraud_risk_score_raw"].clip(0, 100).round(2)


def assign_final_risk_band(score: float) -> str:
    if score >= 80:
        return "critico"
    if score >= 60:
        return "alto"
    if score >= 35:
        return "medio"
    return "baixo"


score_base["final_risk_band"] = score_base["fraud_risk_score"].apply(assign_final_risk_band)
score_base["final_alert"] = score_base["fraud_risk_score"] >= 35

score_base[
    [
        "transacao_id",
        "rule_score",
        "risk_band",
        "fraud_risk_score",
        "final_risk_band",
        "final_alert",
    ]
].head()

,transacao_id,rule_score,risk_band,fraud_risk_score,final_risk_band,final_alert
0,TX_00007707,46,alto,51.7000,medio,True
1,TX_00072064,32,medio,50.9200,medio,True
2,TX_00075764,52,alto,54.5900,medio,True
3,TX_00023203,32,medio,47.1600,medio,True
4,TX_00001302,52,alto,58.4700,medio,True


In [11]:
final_risk_distribution = (
    score_base
    .groupby("final_risk_band")
    .agg(
        qtd_transacoes=("transacao_id", "count"),
        taxa_alerta=("final_alert", "mean"),
        taxa_fraude_sintetica=("is_fraud", "mean"),
        score_medio=("fraud_risk_score", "mean"),
        score_min=("fraud_risk_score", "min"),
        score_max=("fraud_risk_score", "max"),
        valor_medio=("valor", "mean"),
    )
    .reset_index()
)

risk_order = {"baixo": 1, "medio": 2, "alto": 3, "critico": 4}
final_risk_distribution["ordem"] = final_risk_distribution["final_risk_band"].map(risk_order)
final_risk_distribution = final_risk_distribution.sort_values("ordem").drop(columns="ordem")

final_risk_distribution

,final_risk_band,qtd_transacoes,taxa_alerta,taxa_fraude_sintetica,score_medio,score_min,score_max,valor_medio
1,baixo,746,0.0000,0.0000,33.2689,26.7100,34.9900,345.5516
3,medio,67934,1.0000,0.0203,48.3247,35.0000,59.9900,436.7403
0,alto,9738,1.0000,0.4178,66.1264,60.0000,79.9900,"3,464.9805"
2,critico,1582,1.0000,0.9791,85.5382,80.0000,91.8400,"10,958.5807"


## 7. Explicabilidade do Score Final

Nesta etapa será criada uma explicação textual para cada transação.

A explicação combinará:

- regras acionadas;
- score de regras;
- risco estrutural da conta;
- risco estrutural do dispositivo;
- risco estrutural do beneficiário;
- risco estrutural do IP;
- risco da comunidade;
- faixa final de risco.

O objetivo é criar uma saída interpretável para analistas, gestores e avaliadores técnicos.

In [12]:
rule_flag_columns = [
    "R001_alto_valor_transacional",
    "R002_valor_acima_limite_diario",
    "R003_conta_nova_alto_valor",
    "R004_dispositivo_compartilhado",
    "R005_beneficiario_concentrador",
    "R006_rede_ou_device_alto_risco",
    "R007_rajada_transacional_horaria",
    "R008_muitos_beneficiarios_no_dia",
    "R009_ip_compartilhado_multiplas_contas",
    "R010_canal_digital_alto_valor",
]

rule_short_names = {
    "R001_alto_valor_transacional": "alto valor transacional",
    "R002_valor_acima_limite_diario": "valor próximo/acima do limite diário",
    "R003_conta_nova_alto_valor": "conta nova com alto valor",
    "R004_dispositivo_compartilhado": "dispositivo compartilhado",
    "R005_beneficiario_concentrador": "beneficiário concentrador",
    "R006_rede_ou_device_alto_risco": "rede ou dispositivo de alto risco",
    "R007_rajada_transacional_horaria": "rajada transacional horária",
    "R008_muitos_beneficiarios_no_dia": "muitos beneficiários no dia",
    "R009_ip_compartilhado_multiplas_contas": "IP compartilhado por múltiplas contas",
    "R010_canal_digital_alto_valor": "canal digital com alto valor",
}


def list_triggered_rules(row: pd.Series) -> list[str]:
    triggered = []

    for rule_col in rule_flag_columns:
        if rule_col in row.index and bool(row[rule_col]):
            triggered.append(rule_short_names.get(rule_col, rule_col))

    return triggered


def classify_relevant_graph_signals(row: pd.Series) -> list[str]:
    signals = []

    if row["account_graph_entity_graph_risk_score"] >= 70:
        signals.append(
            f"conta com alto risco estrutural ({row['account_graph_entity_graph_risk_score']:.1f})"
        )

    if row["device_graph_entity_graph_risk_score"] >= 70:
        signals.append(
            f"dispositivo com alto risco estrutural ({row['device_graph_entity_graph_risk_score']:.1f})"
        )

    if row["beneficiary_graph_entity_graph_risk_score"] >= 70:
        signals.append(
            f"beneficiário com alto risco estrutural ({row['beneficiary_graph_entity_graph_risk_score']:.1f})"
        )

    if row["ip_graph_entity_graph_risk_score"] >= 70:
        signals.append(
            f"IP com alto risco estrutural ({row['ip_graph_entity_graph_risk_score']:.1f})"
        )

    if row["max_community_risk_score"] >= 70:
        signals.append(
            f"comunidade associada com alto risco ({row['max_community_risk_score']:.1f})"
        )

    return signals


def build_final_explanation(row: pd.Series) -> str:
    triggered_rules = list_triggered_rules(row)
    graph_signals = classify_relevant_graph_signals(row)

    if triggered_rules:
        rules_text = "; ".join(triggered_rules)
    else:
        rules_text = "nenhuma regra transacional crítica acionada"

    if graph_signals:
        graph_text = "; ".join(graph_signals)
    else:
        graph_text = "sem sinal estrutural crítico no grafo"

    return (
        f"Score final {row['fraud_risk_score']:.1f}/100 — risco {row['final_risk_band']}. "
        f"Score de regras: {row['rule_score']:.1f}/100. "
        f"Regras acionadas: {rules_text}. "
        f"Sinais de grafo: {graph_text}."
    )


score_base["triggered_rules_list"] = score_base.apply(list_triggered_rules, axis=1)
score_base["graph_signals_list"] = score_base.apply(classify_relevant_graph_signals, axis=1)
score_base["final_explanation"] = score_base.apply(build_final_explanation, axis=1)

score_base[
    [
        "transacao_id",
        "fraud_scenario",
        "is_fraud",
        "rule_score",
        "fraud_risk_score",
        "final_risk_band",
        "triggered_rules_list",
        "graph_signals_list",
        "final_explanation",
    ]
].head(10)

,transacao_id,fraud_scenario,is_fraud,rule_score,fraud_risk_score,final_risk_band,triggered_rules_list,graph_signals_list,final_explanation
0,TX_00007707,normal,0,46,51.7000,medio,"[dispositivo compartilhado, rede ou dispositivo de alto risco, IP compartilhado por múltiplas contas]",[comunidade associada com alto risco (82.0)],Score final 51.7/100 — risco medio. Score de regras: 46.0/100. Regras acionadas: dispositivo compartilhado; rede ou dispositivo de alto risco; IP compartilhado por múltiplas contas. Sinais de grafo: comunidade associ...
1,TX_00072064,normal,0,32,50.9200,medio,"[dispositivo compartilhado, IP compartilhado por múltiplas contas]","[conta com alto risco estrutural (70.7), dispositivo com alto risco estrutural (73.7), comunidade associada com alto risco (82.0)]",Score final 50.9/100 — risco medio. Score de regras: 32.0/100. Regras acionadas: dispositivo compartilhado; IP compartilhado por múltiplas contas. Sinais de grafo: conta com alto risco estrutural (70.7); dispositivo ...
2,TX_00075764,normal,0,52,54.5900,medio,"[dispositivo compartilhado, beneficiário concentrador, IP compartilhado por múltiplas contas]","[conta com alto risco estrutural (70.6), comunidade associada com alto risco (77.0)]",Score final 54.6/100 — risco medio. Score de regras: 52.0/100. Regras acionadas: dispositivo compartilhado; beneficiário concentrador; IP compartilhado por múltiplas contas. Sinais de grafo: conta com alto risco estr...
3,TX_00023203,normal,0,32,47.1600,medio,"[dispositivo compartilhado, IP compartilhado por múltiplas contas]",[conta com alto risco estrutural (73.7)],Score final 47.2/100 — risco medio. Score de regras: 32.0/100. Regras acionadas: dispositivo compartilhado; IP compartilhado por múltiplas contas. Sinais de grafo: conta com alto risco estrutural (73.7).
4,TX_00001302,normal,0,52,58.4700,medio,"[dispositivo compartilhado, beneficiário concentrador, IP compartilhado por múltiplas contas]","[conta com alto risco estrutural (77.1), beneficiário com alto risco estrutural (74.1), comunidade associada com alto risco (82.0)]",Score final 58.5/100 — risco medio. Score de regras: 52.0/100. Regras acionadas: dispositivo compartilhado; beneficiário concentrador; IP compartilhado por múltiplas contas. Sinais de grafo: conta com alto risco estr...
5,TX_00062859,normal,0,32,38.6700,medio,"[dispositivo compartilhado, IP compartilhado por múltiplas contas]",[comunidade associada com alto risco (82.0)],Score final 38.7/100 — risco medio. Score de regras: 32.0/100. Regras acionadas: dispositivo compartilhado; IP compartilhado por múltiplas contas. Sinais de grafo: comunidade associada com alto risco (82.0).
6,TX_00078572,normal,0,32,47.1800,medio,"[dispositivo compartilhado, IP compartilhado por múltiplas contas]","[IP com alto risco estrutural (70.2), comunidade associada com alto risco (70.4)]",Score final 47.2/100 — risco medio. Score de regras: 32.0/100. Regras acionadas: dispositivo compartilhado; IP compartilhado por múltiplas contas. Sinais de grafo: IP com alto risco estrutural (70.2); comunidade asso...
7,TX_00037550,normal,0,32,40.0900,medio,"[dispositivo compartilhado, IP compartilhado por múltiplas contas]",[comunidade associada com alto risco (82.0)],Score final 40.1/100 — risco medio. Score de regras: 32.0/100. Regras acionadas: dispositivo compartilhado; IP compartilhado por múltiplas contas. Sinais de grafo: comunidade associada com alto risco (82.0).
8,TX_00020042,new_account_high_value,1,100,84.6600,critico,"[alto valor transacional, valor próximo/acima do limite diário, conta nova com alto valor, dispositivo compartilhado, beneficiário concentrador, IP compartilhado por múltiplas contas, canal digital com alto valor]","[conta com alto risco estrutural (86.3), beneficiário com alto risco estrutural (77.7), IP com alto risco estrutural (83.7), comunidade associada com alto risco (77.0)]",Score final 84.7/100 — risco critico. Score de regras: 100.0/100. Regras acionadas: alto valor transacional; valor próx

## 8. Comparação entre Score de Regras e Score Final

Agora será avaliado como o score final se comporta em relação ao score original de regras.

A expectativa é que o score final:

- preserve os sinais fortes do motor de regras;
- aumente a priorização de transações ligadas a entidades estruturalmente suspeitas;
- ajude a identificar transações com risco relacional mesmo quando o score de regras não for extremo.

In [13]:
score_comparison_summary = pd.DataFrame(
    [
        {
            "indicador": "Score médio de regras",
            "valor": score_base["rule_score"].mean(),
        },
        {
            "indicador": "Score final médio",
            "valor": score_base["fraud_risk_score"].mean(),
        },
        {
            "indicador": "Taxa de alerta por regras",
            "valor": score_base["alerta_gerado"].mean(),
        },
        {
            "indicador": "Taxa de alerta final",
            "valor": score_base["final_alert"].mean(),
        },
        {
            "indicador": "Fraude sintética entre alertas por regras",
            "valor": score_base.loc[score_base["alerta_gerado"], "is_fraud"].mean(),
        },
        {
            "indicador": "Fraude sintética entre alertas finais",
            "valor": score_base.loc[score_base["final_alert"], "is_fraud"].mean(),
        },
        {
            "indicador": "Transações críticas por regras",
            "valor": int((score_base["risk_band"] == "critico").sum()),
        },
        {
            "indicador": "Transações críticas pelo score final",
            "valor": int((score_base["final_risk_band"] == "critico").sum()),
        },
    ]
)

score_comparison_summary

,indicador,valor
0,Score médio de regras,44.6321
1,Score final médio,51.0871
2,Taxa de alerta por regras,1.0000
3,Taxa de alerta final,0.9907
4,Fraude sintética entre alertas por regras,0.0875
5,Fraude sintética entre alertas finais,0.0883
6,Transações críticas por regras,"4,627.0000"
7,Transações críticas pelo score final,"1,582.0000"


In [14]:
scenario_score_summary = (
    score_base
    .groupby("fraud_scenario")
    .agg(
        qtd_transacoes=("transacao_id", "count"),
        taxa_fraude_sintetica=("is_fraud", "mean"),
        rule_score_medio=("rule_score", "mean"),
        fraud_risk_score_medio=("fraud_risk_score", "mean"),
        fraud_risk_score_p95=("fraud_risk_score", lambda x: x.quantile(0.95)),
        taxa_alerta_final=("final_alert", "mean"),
        qtd_criticos=("final_risk_band", lambda x: (x == "critico").sum()),
    )
    .reset_index()
    .sort_values("fraud_risk_score_medio", ascending=False)
)

scenario_score_summary

,fraud_scenario,qtd_transacoes,taxa_fraude_sintetica,rule_score_medio,fraud_risk_score_medio,fraud_risk_score_p95,taxa_alerta_final,qtd_criticos
3,coordinated_network,1000,1.0000,94.7900,86.4247,89.7600,1.0000,912
4,new_account_high_value,1000,1.0000,91.7790,75.4559,84.4170,1.0000,243
2,burst_transactions,1200,1.0000,83.0525,72.8850,83.8030,1.0000,242
1,bridge_account,800,1.0000,72.9125,69.4219,85.0630,1.0000,106
0,beneficiary_concentrator,1400,1.0000,61.3064,63.2848,76.2415,1.0000,33
6,shared_device_ring,1600,1.0000,54.4744,62.3070,73.3545,1.0000,13
5,normal,73000,0.0000,41.8222,49.2301,61.5600,0.9898,33


## 9. Priorização Final de Alertas

Nesta etapa será criada a base final de alertas priorizados.

Serão mantidas as transações com `final_alert = True`, ordenadas por:

1. score final;
2. score de regras;
3. risco máximo de entidade no grafo;
4. valor da transação.

Essa saída representa a visão operacional consolidada do MVP.

In [15]:
final_alert_columns = [
    "transacao_id",
    "conta_origem_id",
    "cliente_id",
    "beneficiario_id",
    "device_id",
    "ip_id",
    "valor",
    "data_hora",
    "tipo_transacao",
    "canal",
    "status_transacao",
    "fraud_scenario",
    "is_fraud",
    "qtd_regras_acionadas",
    "rule_score",
    "risk_band",
    "fraud_risk_score",
    "final_risk_band",
    "final_alert",
    "max_entity_graph_risk_score",
    "max_community_risk_score",
    "account_graph_entity_graph_risk_score",
    "device_graph_entity_graph_risk_score",
    "beneficiary_graph_entity_graph_risk_score",
    "ip_graph_entity_graph_risk_score",
    "account_graph_community_id",
    "device_graph_community_id",
    "beneficiary_graph_community_id",
    "ip_graph_community_id",
    "final_explanation",
    *rule_flag_columns,
]

final_alerts = (
    score_base
    .loc[score_base["final_alert"], final_alert_columns]
    .sort_values(
        [
            "fraud_risk_score",
            "rule_score",
            "max_entity_graph_risk_score",
            "valor",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

final_alerts.head(30)

,transacao_id,conta_origem_id,cliente_id,beneficiario_id,device_id,ip_id,valor,data_hora,tipo_transacao,canal,status_transacao,fraud_scenario,is_fraud,qtd_regras_acionadas,rule_score,risk_band,fraud_risk_score,final_risk_band,final_alert,max_entity_graph_risk_score,max_community_risk_score,account_graph_entity_graph_risk_score,device_graph_entity_graph_risk_score,beneficiary_graph_entity_graph_risk_score,ip_graph_entity_graph_risk_score,account_graph_community_id,device_graph_community_id,beneficiary_graph_community_id,ip_graph_community_id,final_explanation,R001_alto_valor_transacional,R002_valor_acima_limite_diario,R003_conta_nova_alto_valor,R004_dispositivo_compartilhado,R005_beneficiario_concentrador,R006_rede_ou_device_alto_risco,R007_rajada_transacional_horaria,R008_muitos_beneficiarios_no_dia,R009_ip_compartilhado_multiplas_contas,R010_canal_digital_alto_valor
0,TX_00001194,CTA_000875,CLI_003349,BEN_000475,DEV_000361,IP_000962,"4,097.7900",2025-01-31 05:20:18,transferencia_interna,agencia,aprovada,bridge_account,1,6,100,critico,91.8400,critico,True,95.1300,74.8000,95.1300,89.3600,68.0400,87.9300,4,4,4,7,Score final 91.8/100 — risco critico. Score de regras: 100.0/100. Regras acionadas: alto valor transacional; conta nova com alto valor; dispositivo compartilhado; beneficiário concentrador; rede ou dispositivo de alt...,True,False,True,True,True,True,False,False,True,False
1,TX_00062430,CTA_003281,CLI_000859,BEN_000283,DEV_000847,IP_002363,"2,133.9200",2025-11-20 21:37:00,pix,app,em_analise,burst_transactions,1,8,100,critico,91.7200,critico,True,96.2800,82.0100,95.2000,70.4300,96.2800,54.8300,12,12,16,7,Score final 91.7/100 — risco critico. Score de regras: 100.0/100. Regras acionadas: valor próximo/acima do limite diário; dispositivo compartilhado; beneficiário concentrador; rede ou dispositivo de alto risco; rajad...,False,True,False,True,True,True,True,True,True,True
2,TX_00059542,CTA_003015,CLI_004810,BEN_000010,DEV_000671,IP_002218,"18,751.9400",2025-07-12 04:42:51,pix,app,negada,new_account_high_value,1,8,100,critico,91.5300,critico,True,95.7400,82.0100,88.8400,95.7400,70.8800,76.6300,16,2,16,16,Score final 91.5/100 — risco critico. Score de regras: 100.0/100. Regras acionadas: alto valor transacional; valor próximo/acima do limite diário; conta nova com alto valor; dispositivo compartilhado; beneficiário co...,True,True,True,True,True,True,False,False,True,True
3,TX_00050078,CTA_005360,CLI_002013,BEN_002660,DEV_000105,IP_001775,"10,123.4800",2025-02-25 08:09:15,pix,api,em_analise,bridge_account,1,8,100,critico,91.4700,critico,True,94.7500,74.8000,93.3200,94.7500,65.3500,81.0000,21,7,21,12,Score final 91.5/100 — risco critico. Score de regras: 100.0/100. Regras acionadas: alto valor transacional; valor próximo/acima do limite diário; conta nova com alto valor; dispositivo compartilhado; beneficiário co...,True,True,True,True,True,True,False,False,True,True
4,TX_00005110,CTA_003572,CLI_002305,BEN_002766,DEV_002397,IP_001627,"1,095.2100",2025-11-20 21:09:00,pix,app,aprovada,burst_transactions,1,6,100,critico,90.5800,critico,True,89.3900,82.0100,89.3900,87.7700,65.4900,88.9700,1,10,16,12,Score final 90.6/100 — risco critico. Score de regras: 100.0/100. Regras acionadas: dispositivo compartilhado; beneficiário concentrador; rede ou dispositivo de alto risco; rajada transacional horária; muitos benefic...,False,False,False,True,True,True,True,True,True,False
5,TX_00048598,CTA_002863,CLI_004945,BEN_000221,DEV_004329,IP_002713,"3,973.7000",2025-08-05 11:46:31,boleto,app,aprovada,coordinated_network,1,8,100,critico,90.3400,critico,True,89.2800,44.4100,89.2800,84.1700,85.2000,81.7600,8,8,8,8,Score final 90.3/100 — risco critico. Score de regras: 100.0/100. Regras acionadas: alto valor transacional; valor próximo/acima do limite diário; conta nova com alto valor; dispositivo compartilhado; beneficiário co...,True,True,True,True,True,True,False,False,True,True
6,TX_00013631,CTA_002741,CLI_004122,BEN_000221,DEV_001936,IP_00199

In [16]:
critical_final_alerts = (
    final_alerts
    .loc[final_alerts["final_risk_band"] == "critico"]
    .copy()
)

critical_final_alerts[
    [
        "transacao_id",
        "conta_origem_id",
        "valor",
        "fraud_scenario",
        "rule_score",
        "fraud_risk_score",
        "final_risk_band",
        "max_entity_graph_risk_score",
        "max_community_risk_score",
        "final_explanation",
    ]
].head(20)

,transacao_id,conta_origem_id,valor,fraud_scenario,rule_score,fraud_risk_score,final_risk_band,max_entity_graph_risk_score,max_community_risk_score,final_explanation
0,TX_00001194,CTA_000875,"4,097.7900",bridge_account,100,91.8400,critico,95.1300,74.8000,Score final 91.8/100 — risco critico. Score de regras: 100.0/100. Regras acionadas: alto valor transacional; conta nova com alto valor; dispositivo compartilhado; beneficiário concentrador; rede ou dispositivo de alt...
1,TX_00062430,CTA_003281,"2,133.9200",burst_transactions,100,91.7200,critico,96.2800,82.0100,Score final 91.7/100 — risco critico. Score de regras: 100.0/100. Regras acionadas: valor próximo/acima do limite diário; dispositivo compartilhado; beneficiário concentrador; rede ou dispositivo de alto risco; rajad...
2,TX_00059542,CTA_003015,"18,751.9400",new_account_high_value,100,91.5300,critico,95.7400,82.0100,Score final 91.5/100 — risco critico. Score de regras: 100.0/100. Regras acionadas: alto valor transacional; valor próximo/acima do limite diário; conta nova com alto valor; dispositivo compartilhado; beneficiário co...
3,TX_00050078,CTA_005360,"10,123.4800",bridge_account,100,91.4700,critico,94.7500,74.8000,Score final 91.5/100 — risco critico. Score de regras: 100.0/100. Regras acionadas: alto valor transacional; valor próximo/acima do limite diário; conta nova com alto valor; dispositivo compartilhado; beneficiário co...
4,TX_00005110,CTA_003572,"1,095.2100",burst_transactions,100,90.5800,critico,89.3900,82.0100,Score final 90.6/100 — risco critico. Score de regras: 100.0/100. Regras acionadas: dispositivo compartilhado; beneficiário concentrador; rede ou dispositivo de alto risco; rajada transacional horária; muitos benefic...
5,TX_00048598,CTA_002863,"3,973.7000",coordinated_network,100,90.3400,critico,89.2800,44.4100,Score final 90.3/100 — risco critico. Score de regras: 100.0/100. Regras acionadas: alto valor transacional; valor próximo/acima do limite diário; conta nova com alto valor; dispositivo compartilhado; beneficiário co...
6,TX_00013631,CTA_002741,"17,374.6800",coordinated_network,100,90.3100,critico,89.4300,44.4100,Score final 90.3/100 — risco critico. Score de regras: 100.0/100. Regras acionadas: alto valor transacional; valor próximo/acima do limite diário; dispositivo compartilhado; beneficiário concentrador; rede ou disposi...
7,TX_00010873,CTA_002863,"10,070.3600",coordinated_network,100,90.2500,critico,89.2800,44.4100,Score final 90.2/100 — risco critico. Score de regras: 100.0/100. Regras acionadas: alto valor transacional; valor próximo/acima do limite diário; conta nova com alto valor; dispositivo compartilhado; beneficiário co...
8,TX_00055456,CTA_004244,"6,321.1000",coordinated_network,100,90.1800,critico,88.5500,44.4100,Score final 90.2/100 — risco critico. Score de regras: 100.0/100. Regras acionadas: alto valor transacional; valor próximo/acima do limite diário; dispositivo compartilhado; beneficiário concentrador; rede ou disposi...
9,TX_00068130,CTA_003481,"5,914.8700",coordinated_network,100,90.1200,critico,89.5300,44.4100,Score final 90.1/100 — risco critico. Score de regras: 100.0/100. Regras acionadas: alto valor transacional; valor próximo/acima do limite diário; dispositivo compartilhado; beneficiário concentrador; rede ou disposi...


## 10. Visões Agregadas para Investigação

Além do alerta por transação, é útil criar visões agregadas por:

- conta;
- dispositivo;
- beneficiário;
- IP;
- comunidade.

Essas visões ajudam a priorizar investigações em nível de entidade e rede.

In [17]:
account_final_summary = (
    final_alerts
    .groupby("conta_origem_id")
    .agg(
        qtd_alertas=("transacao_id", "count"),
        score_final_medio=("fraud_risk_score", "mean"),
        score_final_max=("fraud_risk_score", "max"),
        valor_total_alertado=("valor", "sum"),
        qtd_beneficiarios=("beneficiario_id", "nunique"),
        qtd_devices=("device_id", "nunique"),
        qtd_ips=("ip_id", "nunique"),
        taxa_fraude_sintetica=("is_fraud", "mean"),
        max_entity_graph_risk_score=("max_entity_graph_risk_score", "max"),
        max_community_risk_score=("max_community_risk_score", "max"),
    )
    .reset_index()
    .sort_values(["score_final_max", "qtd_alertas", "valor_total_alertado"], ascending=False)
)

device_final_summary = (
    final_alerts
    .groupby("device_id")
    .agg(
        qtd_alertas=("transacao_id", "count"),
        contas_distintas=("conta_origem_id", "nunique"),
        score_final_medio=("fraud_risk_score", "mean"),
        score_final_max=("fraud_risk_score", "max"),
        valor_total_alertado=("valor", "sum"),
        taxa_fraude_sintetica=("is_fraud", "mean"),
    )
    .reset_index()
    .sort_values(["contas_distintas", "score_final_max", "qtd_alertas"], ascending=False)
)

beneficiary_final_summary = (
    final_alerts
    .groupby("beneficiario_id")
    .agg(
        qtd_alertas=("transacao_id", "count"),
        contas_origem_distintas=("conta_origem_id", "nunique"),
        score_final_medio=("fraud_risk_score", "mean"),
        score_final_max=("fraud_risk_score", "max"),
        valor_total_alertado=("valor", "sum"),
        taxa_fraude_sintetica=("is_fraud", "mean"),
    )
    .reset_index()
    .sort_values(["contas_origem_distintas", "score_final_max", "qtd_alertas"], ascending=False)
)

ip_final_summary = (
    final_alerts
    .groupby("ip_id")
    .agg(
        qtd_alertas=("transacao_id", "count"),
        contas_distintas=("conta_origem_id", "nunique"),
        score_final_medio=("fraud_risk_score", "mean"),
        score_final_max=("fraud_risk_score", "max"),
        valor_total_alertado=("valor", "sum"),
        taxa_fraude_sintetica=("is_fraud", "mean"),
    )
    .reset_index()
    .sort_values(["contas_distintas", "score_final_max", "qtd_alertas"], ascending=False)
)

display(account_final_summary.head(15))
display(device_final_summary.head(15))
display(beneficiary_final_summary.head(15))
display(ip_final_summary.head(15))

,conta_origem_id,qtd_alertas,score_final_medio,score_final_max,valor_total_alertado,qtd_beneficiarios,qtd_devices,qtd_ips,taxa_fraude_sintetica,max_entity_graph_risk_score,max_community_risk_score
873,CTA_000875,75,66.9080,91.8400,"385,964.2600",75,74,73,0.8400,95.1300,82.0100
3279,CTA_003281,55,70.7345,91.7200,"92,931.9300",54,55,55,0.6909,96.2800,82.0100
3013,CTA_003015,19,60.8421,91.5300,"124,621.0000",19,19,19,0.2632,95.7400,82.0100
5358,CTA_005360,79,67.6987,91.4700,"409,791.8000",78,77,78,0.8354,94.7500,82.0100
3570,CTA_003572,39,66.6992,90.5800,"60,862.8500",39,39,39,0.6667,90.5400,82.0100
2861,CTA_002863,33,67.1615,90.3400,"115,095.3600",29,27,26,0.4242,89.2800,82.0100
2739,CTA_002741,32,70.3044,90.3100,"193,349.1900",25,22,24,0.5312,89.4300,82.0100
4242,CTA_004244,32,72.8588,90.1800,"179,777.0900",23,22,24,0.5625,88.5500,82.0100
322,CTA_000323,34,73.0962,90.1200,"157,852.1000",28,25,25,0.5882,87.3700,82.0100
3479,CTA_003481,32,69.3109,90.1200,"149,351.5300",27,24,25,0.4688,96.2800,82.0100


,device_id,qtd_alertas,contas_distintas,score_final_medio,score_final_max,valor_total_alertado,taxa_fraude_sintetica
855,DEV_000856,150,147,63.3721,84.4100,"74,310.8300",0.8933
4006,DEV_004007,145,143,62.1557,89.0400,"60,139.8700",0.9241
3931,DEV_003932,100,99,62.8082,86.3500,"46,842.8900",0.8100
2417,DEV_002418,97,96,62.5267,75.7600,"31,917.1200",0.8351
2047,DEV_002048,95,91,63.3573,89.1000,"55,695.3100",0.7474
101,DEV_000102,89,88,62.6945,79.3100,"40,333.0900",0.7978
2458,DEV_002459,88,88,62.3983,78.8600,"41,954.5400",0.7727
1271,DEV_001272,87,86,61.7605,88.4600,"82,371.4700",0.8276
3862,DEV_003863,86,86,62.6310,79.9900,"34,586.0800",0.8372
360,DEV_000361,83,83,62.9540,91.8400,"55,482.9900",0.7108


,beneficiario_id,qtd_alertas,contas_origem_distintas,score_final_medio,score_final_max,valor_total_alertado,taxa_fraude_sintetica
2998,BEN_002999,108,106,64.6209,87.8700,"180,489.2200",0.7870
3122,BEN_003123,108,105,62.5851,81.8500,"140,199.2600",0.7315
3030,BEN_003031,103,102,63.5499,85.2700,"185,369.5200",0.8058
267,BEN_000268,103,102,62.2286,81.7500,"136,141.7500",0.8447
1263,BEN_001264,97,97,61.5532,78.9400,"127,914.9900",0.7216
1790,BEN_001791,95,95,62.4224,84.4500,"126,121.5500",0.7474
351,BEN_000352,94,94,62.4589,80.5300,"140,127.4400",0.8511
2786,BEN_002787,94,93,63.6515,83.1400,"152,518.1000",0.7553
997,BEN_000998,93,93,61.0788,80.4500,"95,672.9300",0.7849
2456,BEN_002457,93,93,60.9528,79.9300,"105,149.3000",0.7204


,ip_id,qtd_alertas,contas_distintas,score_final_medio,score_final_max,valor_total_alertado,taxa_fraude_sintetica
2368,IP_002369,134,92,77.5030,90.0200,"934,404.9100",0.7015
1996,IP_001997,145,91,79.1777,90.3100,"1,033,160.5500",0.7655
1100,IP_001101,140,79,80.6794,90.1200,"1,048,210.3900",0.8143
1108,IP_001109,144,78,81.4255,90.1800,"1,108,719.6800",0.8472
563,IP_000564,128,75,80.3556,90.0700,"946,469.3200",0.7969
2712,IP_002713,124,74,79.3744,90.3400,"920,033.6400",0.7661
567,IP_000568,120,71,81.5755,90.2500,"963,521.4000",0.8417
2422,IP_002423,117,70,80.7030,89.9200,"798,075.4600",0.8462
1279,IP_001280,115,69,82.5914,89.8900,"1,003,049.4900",0.8522
479,IP_000480,101,61,81.0048,90.0900,"839,086.9800",0.8416


## 11. Síntese Executiva do Score Final

Nesta etapa será criada uma síntese executiva com os principais indicadores do score final.

Essa tabela será usada no relatório do notebook e, futuramente, no README premium e na página de portfólio.

In [18]:
final_executive_summary = pd.DataFrame(
    [
        {
            "indicador": "Total de transações avaliadas",
            "valor": len(score_base),
            "interpretacao": "Quantidade total de eventos avaliados pelo score final.",
        },
        {
            "indicador": "Total de alertas finais",
            "valor": int(score_base["final_alert"].sum()),
            "interpretacao": "Transações priorizadas pela combinação de regras e grafo.",
        },
        {
            "indicador": "Taxa de alertas finais",
            "valor": score_base["final_alert"].mean(),
            "interpretacao": "Proporção da base direcionada para investigação.",
        },
        {
            "indicador": "Score final médio",
            "valor": score_base["fraud_risk_score"].mean(),
            "interpretacao": "Risco médio após combinação de sinais transacionais e relacionais.",
        },
        {
            "indicador": "Alertas críticos finais",
            "valor": int((score_base["final_risk_band"] == "critico").sum()),
            "interpretacao": "Transações com score final igual ou superior a 80.",
        },
        {
            "indicador": "Taxa sintética de fraude entre alertas finais",
            "valor": final_alerts["is_fraud"].mean(),
            "interpretacao": "Validação exploratória contra o label sintético.",
        },
        {
            "indicador": "Contas com alertas finais",
            "valor": final_alerts["conta_origem_id"].nunique(),
            "interpretacao": "Quantidade de contas distintas priorizadas.",
        },
        {
            "indicador": "Dispositivos com alertas finais",
            "valor": final_alerts["device_id"].nunique(),
            "interpretacao": "Quantidade de dispositivos distintos envolvidos.",
        },
        {
            "indicador": "Beneficiários com alertas finais",
            "valor": final_alerts["beneficiario_id"].nunique(),
            "interpretacao": "Quantidade de beneficiários distintos envolvidos.",
        },
        {
            "indicador": "IPs com alertas finais",
            "valor": final_alerts["ip_id"].nunique(),
            "interpretacao": "Quantidade de IPs distintos envolvidos.",
        },
    ]
)

final_executive_summary

,indicador,valor,interpretacao
0,Total de transações avaliadas,"80,000.0000",Quantidade total de eventos avaliados pelo score final.
1,Total de alertas finais,"79,254.0000",Transações priorizadas pela combinação de regras e grafo.
2,Taxa de alertas finais,0.9907,Proporção da base direcionada para investigação.
3,Score final médio,51.0871,Risco médio após combinação de sinais transacionais e relacionais.
4,Alertas críticos finais,"1,582.0000",Transações com score final igual ou superior a 80.
5,Taxa sintética de fraude entre alertas finais,0.0883,Validação exploratória contra o label sintético.
6,Contas com alertas finais,"5,999.0000",Quantidade de contas distintas priorizadas.
7,Dispositivos com alertas finais,"4,500.0000",Quantidade de dispositivos distintos envolvidos.
8,Beneficiários com alertas finais,"3,500.0000",Quantidade de beneficiários distintos envolvidos.
9,IPs com alertas finais,"3,000.0000",Quantidade de IPs distintos envolvidos.


## 12. Interpretação Analítica

Nesta etapa serão registrados os principais aprendizados do score final.

O objetivo é transformar resultados técnicos em narrativa de negócio para prevenção a fraudes.

In [19]:
score_findings = pd.DataFrame(
    [
        {
            "achado": "O score final combina risco transacional e risco relacional",
            "evidencia": "O score utiliza rule_score, risco de conta, dispositivo, beneficiário, IP e comunidade.",
            "interpretacao": "A priorização deixa de depender apenas da transação isolada e passa a considerar o contexto da rede.",
        },
        {
            "achado": "Entidades estruturalmente suspeitas elevam a prioridade investigativa",
            "evidencia": "Transações conectadas a entidades com alto entity_graph_risk_score recebem reforço no score final.",
            "interpretacao": "Dispositivos compartilhados, beneficiários concentradores e IPs recorrentes tornam-se sinais complementares.",
        },
        {
            "achado": "Comunidades de grafo agregam contexto coletivo",
            "evidencia": "O max_community_risk_score incorpora o risco das comunidades associadas às entidades da transação.",
            "interpretacao": "A análise passa a observar grupos conectados, não apenas eventos pontuais.",
        },
        {
            "achado": "A explicabilidade melhora a utilidade operacional",
            "evidencia": "Cada transação recebe uma justificativa textual com regras acionadas e sinais de grafo.",
            "interpretacao": "A saída pode apoiar triagem, revisão manual, comunicação executiva e documentação do raciocínio.",
        },
        {
            "achado": "A abordagem é adequada para portfólio sênior",
            "evidencia": "O projeto integra CRISP-DM+, regras, grafos, score e explicabilidade em uma trilha reprodutível.",
            "interpretacao": "Demonstra capacidade de transformar problema de negócio em solução analítica estruturada.",
        },
    ]
)

score_findings

,achado,evidencia,interpretacao
0,O score final combina risco transacional e risco relacional,"O score utiliza rule_score, risco de conta, dispositivo, beneficiário, IP e comunidade.",A priorização deixa de depender apenas da transação isolada e passa a considerar o contexto da rede.
1,Entidades estruturalmente suspeitas elevam a prioridade investigativa,Transações conectadas a entidades com alto entity_graph_risk_score recebem reforço no score final.,"Dispositivos compartilhados, beneficiários concentradores e IPs recorrentes tornam-se sinais complementares."
2,Comunidades de grafo agregam contexto coletivo,O max_community_risk_score incorpora o risco das comunidades associadas às entidades da transação.,"A análise passa a observar grupos conectados, não apenas eventos pontuais."
3,A explicabilidade melhora a utilidade operacional,Cada transação recebe uma justificativa textual com regras acionadas e sinais de grafo.,"A saída pode apoiar triagem, revisão manual, comunicação executiva e documentação do raciocínio."
4,A abordagem é adequada para portfólio sênior,"O projeto integra CRISP-DM+, regras, grafos, score e explicabilidade em uma trilha reprodutível.",Demonstra capacidade de transformar problema de negócio em solução analítica estruturada.


## 13. Exportação dos Artefatos

Nesta etapa serão exportados os principais resultados:

- base final com score de risco;
- alertas finais priorizados;
- resumos agregados;
- amostras em CSV;
- metodologia do score;
- relatório executivo.

In [20]:
final_score_path = GOLD_DIR / "final_fraud_risk_score.parquet"
final_alerts_path = GOLD_DIR / "final_fraud_alerts.parquet"

account_final_summary_path = GOLD_DIR / "account_final_risk_summary.parquet"
device_final_summary_path = GOLD_DIR / "device_final_risk_summary.parquet"
beneficiary_final_summary_path = GOLD_DIR / "beneficiary_final_risk_summary.parquet"
ip_final_summary_path = GOLD_DIR / "ip_final_risk_summary.parquet"

final_alerts_sample_csv_path = REPORTS_DIR / "final_fraud_alerts_sample.csv"
critical_final_alerts_csv_path = REPORTS_DIR / "critical_final_fraud_alerts.csv"
score_comparison_csv_path = REPORTS_DIR / "score_comparison_summary.csv"
scenario_score_summary_csv_path = REPORTS_DIR / "scenario_score_summary.csv"

score_base.to_parquet(final_score_path, index=False)
final_alerts.to_parquet(final_alerts_path, index=False)

account_final_summary.to_parquet(account_final_summary_path, index=False)
device_final_summary.to_parquet(device_final_summary_path, index=False)
beneficiary_final_summary.to_parquet(beneficiary_final_summary_path, index=False)
ip_final_summary.to_parquet(ip_final_summary_path, index=False)

final_alerts.head(200).to_csv(final_alerts_sample_csv_path, index=False, encoding="utf-8")
critical_final_alerts.head(200).to_csv(critical_final_alerts_csv_path, index=False, encoding="utf-8")
score_comparison_summary.to_csv(score_comparison_csv_path, index=False, encoding="utf-8")
scenario_score_summary.to_csv(scenario_score_summary_csv_path, index=False, encoding="utf-8")

print("Artefatos exportados com sucesso:")
print(f"- {final_score_path}")
print(f"- {final_alerts_path}")
print(f"- {final_alerts_sample_csv_path}")
print(f"- {critical_final_alerts_csv_path}")
print(f"- {score_comparison_csv_path}")
print(f"- {scenario_score_summary_csv_path}")

Artefatos exportados com sucesso:
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\data\03-gold\final_fraud_risk_score.parquet
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\data\03-gold\final_fraud_alerts.parquet
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\artifacts\reports\final_fraud_alerts_sample.csv
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\artifacts\reports\critical_final_fraud_alerts.csv
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\artifacts\reports\score_comparison_summary.csv
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\artifacts\reports\scenario_score_summary.csv


In [21]:
def safe_markdown_table(df: pd.DataFrame) -> str:
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return df.to_csv(index=False)


score_methodology_lines = [
    "# Metodologia do Fraud Risk Score",
    "",
    "Este documento descreve a metodologia usada no Notebook 06 para composição do score final de risco antifraude.",
    "",
    "## Componentes do Score",
    "",
    safe_markdown_table(score_methodology),
    "",
    "## Fórmula Conceitual",
    "",
    "```text",
    "fraud_risk_score =",
    "    0.45  * rule_score",
    "  + 0.20  * account_graph_risk_score",
    "  + 0.125 * device_graph_risk_score",
    "  + 0.125 * beneficiary_graph_risk_score",
    "  + 0.05  * ip_graph_risk_score",
    "  + 0.05  * max_community_risk_score",
    "```",
    "",
    "## Faixas de Risco",
    "",
    "| Faixa | Critério |",
    "|---|---|",
    "| crítico | score >= 80 |",
    "| alto | 60 <= score < 80 |",
    "| médio | 35 <= score < 60 |",
    "| baixo | score < 35 |",
    "",
    "## Observação",
    "",
    "O score foi criado para fins educacionais, analíticos e de portfólio. Ele não representa um modelo produtivo de decisão antifraude.",
    "",
]

score_methodology_path = DOCS_DIR / "score_methodology.md"
score_methodology_path.write_text("\n".join(score_methodology_lines), encoding="utf-8")

print(f"Metodologia salva em: {score_methodology_path}")

Metodologia salva em: d:\_DS-Projects\Data-Science\fraud-graph-analytics\docs\score_methodology.md


In [22]:
score_report_lines = [
    "# Fraud Risk Score Explainability — Resumo Executivo",
    "",
    "Este documento consolida os principais resultados do Notebook 06.",
    "",
    "## Objetivo",
    "",
    "Combinar regras antifraude explicáveis com features estruturais de grafo para criar um score final de risco transacional.",
    "",
    "## Síntese Executiva",
    "",
    safe_markdown_table(final_executive_summary),
    "",
    "## Distribuição por Faixa de Risco Final",
    "",
    safe_markdown_table(final_risk_distribution),
    "",
    "## Comparação entre Score de Regras e Score Final",
    "",
    safe_markdown_table(score_comparison_summary),
    "",
    "## Score por Cenário Sintético",
    "",
    safe_markdown_table(scenario_score_summary),
    "",
    "## Top Contas por Alertas Finais",
    "",
    safe_markdown_table(account_final_summary.head(10)),
    "",
    "## Top Dispositivos por Alertas Finais",
    "",
    safe_markdown_table(device_final_summary.head(10)),
    "",
    "## Top Beneficiários por Alertas Finais",
    "",
    safe_markdown_table(beneficiary_final_summary.head(10)),
    "",
    "## Top IPs por Alertas Finais",
    "",
    safe_markdown_table(ip_final_summary.head(10)),
    "",
    "## Principais Achados",
    "",
    safe_markdown_table(score_findings),
    "",
    "## Valor Analítico",
    "",
    "- Integra regras transacionais e contexto relacional de grafo.",
    "- Prioriza transações com maior risco final.",
    "- Explica os motivos do alerta em linguagem interpretável.",
    "- Permite investigação por transação, conta, dispositivo, beneficiário, IP e comunidade.",
    "- Prepara o projeto para conclusão executiva e empacotamento de portfólio.",
    "",
    "## Observação",
    "",
    "Os resultados são derivados de dados sintéticos criados exclusivamente para fins educacionais, analíticos e de portfólio.",
    "",
]

score_report_path = DOCS_DIR / "fraud_risk_score_explainability_summary.md"
score_report_path.write_text("\n".join(score_report_lines), encoding="utf-8")

print(f"Relatório executivo salvo em: {score_report_path}")

Relatório executivo salvo em: d:\_DS-Projects\Data-Science\fraud-graph-analytics\docs\fraud_risk_score_explainability_summary.md


## 14. Conclusão Executiva do Notebook 06

Este notebook construiu o **Fraud Risk Score final** do projeto Fraud Graph Analytics.

A abordagem integrou:

- score de regras antifraude;
- risco estrutural da conta;
- risco estrutural do dispositivo;
- risco estrutural do beneficiário;
- risco estrutural do IP;
- risco da comunidade;
- explicabilidade textual por transação.

O resultado é uma camada de priorização antifraude que combina sinais transacionais e relacionais, permitindo investigar eventos suspeitos com mais contexto.

As principais saídas deste notebook foram:

- `final_fraud_risk_score.parquet`;
- `final_fraud_alerts.parquet`;
- `final_fraud_alerts_sample.csv`;
- `critical_final_fraud_alerts.csv`;
- `score_methodology.md`;
- `fraud_risk_score_explainability_summary.md`.

O próximo passo será o:

**Notebook 07 — Executive Conclusion & Portfolio Packaging**

Nesse notebook, o projeto será consolidado em uma narrativa executiva para GitHub, portfólio e apresentação profissional.